<a href="https://colab.research.google.com/github/Fawada/AI-ML-course-notebooks/blob/main/module2/m2_notebook3_data_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Module 2 · Lesson 2 Notebook
# Data, Labels, and Evaluation in Practice

---

**Module 2 · Deep Neural Networks**  
**Estimated time:** 45 minutes  
**Level:** Beginner — assumes you have read the Lesson 2 reading pages on data pipelines and evaluation metrics

---

## 📋 What This Notebook Does

The Lesson 2 reading covered the full data lifecycle — collection, cleaning, labelling, splitting, and monitoring — and then introduced the evaluation metrics that go beyond simple accuracy: precision, recall, F1, ROC/AUC, and regression metrics.

This notebook puts all of that into practice. You will work with a realistic stellar dataset that has been deliberately constructed to contain the kinds of problems real data has — class imbalance, outliers, and noisy features. You will clean it, split it properly, train a classifier, and then evaluate it using the full range of metrics from the reading.

By the end of this notebook you will have:

- Explored and diagnosed data quality issues: missing values, class imbalance, and outliers
- Applied a full data preprocessing pipeline: cleaning, normalisation, and stratified splitting
- Trained a classifier and computed accuracy, precision, recall, F1, and AUC
- Seen concretely why accuracy is misleading on imbalanced data
- Explored the precision–recall trade-off interactively by adjusting the classification threshold
- Computed regression metrics (MAE, RMSE, R²) on a continuous prediction task

---

## 🗺️ Notebook Structure

| Step | What you do |
|------|-------------|
| **0. Setup** | Install and import libraries |
| **1. Load and Explore** | Load the dataset and diagnose data quality problems |
| **2. Clean and Preprocess** | Fix the problems: handle missing values, outliers, imbalance |
| **3. Split the Data** | Stratified train/validation/test split |
| **4. Classification Metrics** | Train a classifier and compute precision, recall, F1, AUC |
| **5. Why Accuracy Misleads** | See the imbalanced-data accuracy trap in action |
| **6. Precision–Recall Trade-off** | Adjust the decision threshold interactively |
| **7. Regression Metrics** | Predict stellar temperature and compute MAE, RMSE, R² |
| **8. Exercises** | Guided reflection |

---

> **Before you start:** Read the Lesson 2 pages on Data Pipelines and Metrics Beyond Accuracy. Every concept in those pages appears directly in the code below — the reading gives you the *why*, this notebook gives you the *how*.

---
## Step 0 — Setup

We will use:
- `numpy` and `pandas` for data handling
- `matplotlib` and `seaborn` for visualisation
- `scikit-learn` for preprocessing, model training, and all evaluation metrics
- `ipywidgets` for the interactive precision–recall threshold explorer

All libraries are pre-installed on Google Colab.

In [ ]:
!pip install ipywidgets --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score,
    precision_recall_curve
)

import ipywidgets as widgets
from ipywidgets import interact

np.random.seed(42)
sns.set(style='whitegrid')
%matplotlib inline

print('✅ All libraries imported successfully.')

---
## Step 1 — Load and Explore the Data

### 1.1 — About the Dataset

We are working with a **simulated stellar survey dataset** containing 5,000 observations of celestial objects. Each row represents one object with the following features:

| Feature | Description |
|---------|-------------|
| `u`, `g`, `r`, `i`, `z` | Photometric magnitudes in five filters |
| `redshift` | Spectral redshift (a proxy for distance) |
| `temperature_K` | Estimated stellar temperature in Kelvin |
| `snr` | Signal-to-noise ratio of the observation |
| `object_class` | Target label: **STAR**, **GALAXY**, or **QSO** |

**This dataset has been deliberately constructed to contain real-world data quality problems:**
- Some feature values are missing
- Some measurements contain outliers from instrument errors
- The class distribution is imbalanced — QSOs are rare

Your job in Steps 1–3 is to find and fix these problems before any modelling begins.

In [ ]:
np.random.seed(42)
n_total = 5000

# Class sizes — deliberately imbalanced (mirrors real sky survey distributions)
n_star   = 2500   # 50%
n_galaxy = 2000   # 40%
n_qso    = 500    # 10% — QSOs are genuinely rare

def make_class(n, u_mean, g_mean, r_mean, z_mean, redshift_mean, redshift_std, temp_mean, temp_std):
    u         = np.random.normal(u_mean,        0.9,          n)
    g         = np.random.normal(g_mean,        0.8,          n)
    r         = np.random.normal(r_mean,        0.7,          n)
    i         = np.random.normal(r_mean - 0.3,  0.6,          n)
    z         = np.random.normal(z_mean,        0.5,          n)
    redshift  = np.abs(np.random.normal(redshift_mean, redshift_std, n))
    temp      = np.random.normal(temp_mean,     temp_std,     n)
    snr       = np.random.exponential(15, n) + 5
    return np.column_stack([u, g, r, i, z, redshift, temp, snr])

X_star   = make_class(n_star,   19.5, 18.8, 18.3, 17.8, 0.0002, 0.0001, 5800, 800)
X_galaxy = make_class(n_galaxy, 21.5, 20.5, 19.8, 18.9, 0.08,   0.04,   4200, 600)
X_qso    = make_class(n_qso,    20.0, 19.8, 19.6, 19.0, 1.5,    0.8,    15000, 3000)

X_all = np.vstack([X_star, X_galaxy, X_qso])
y_all = np.array(['STAR']*n_star + ['GALAXY']*n_galaxy + ['QSO']*n_qso)

cols = ['u','g','r','i','z','redshift','temperature_K','snr']
df   = pd.DataFrame(X_all, columns=cols)
df['object_class'] = y_all

# ── Inject realistic data quality problems ──
# 1. Missing values (~4% of cells in selected columns)
for col in ['u', 'g', 'snr']:
    mask = np.random.choice([True, False], size=len(df), p=[0.04, 0.96])
    df.loc[mask, col] = np.nan

# 2. Instrument outliers in temperature (rare but extreme)
outlier_idx = np.random.choice(len(df), size=40, replace=False)
df.loc[outlier_idx, 'temperature_K'] = np.random.uniform(80000, 200000, 40)

# 3. Shuffle rows
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset shape: {df.shape}  ({df.shape[0]} objects, {df.shape[1]} columns)')
print(f'\nFirst 5 rows:')
df.head()

### 1.2 — Diagnose Data Quality Problems

Before doing any cleaning, we need to understand exactly what problems exist. The reading described three key diagnostic tasks: checking for missing data, checking for unusual values, and checking the class distribution. Run the cells below to perform all three.

In [ ]:
# --- Check 1: Missing values ---
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing count': missing, 'Missing %': missing_pct})
print(missing_report[missing_report['Missing count'] > 0])

print(f'\nTotal cells: {df.shape[0] * df.shape[1]:,}')
print(f'Missing cells: {missing.sum():,}  ({missing.sum() / (df.shape[0]*df.shape[1])*100:.2f}%)')

In [ ]:
# --- Check 2: Class distribution (imbalance) ---
print('=== CLASS DISTRIBUTION ===')
class_counts = df['object_class'].value_counts()
class_pct    = (class_counts / len(df) * 100).round(1)
for cls, cnt, pct in zip(class_counts.index, class_counts.values, class_pct.values):
    bar = '█' * int(pct / 2)
    print(f'  {cls:8s}  {cnt:5d} ({pct:5.1f}%)  {bar}')

fig, ax = plt.subplots(figsize=(7, 4))
colors = {'STAR': '#2196F3', 'GALAXY': '#4CAF50', 'QSO': '#FF5722'}
bars = ax.bar(class_counts.index, class_counts.values,
              color=[colors[c] for c in class_counts.index], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            str(val), ha='center', fontsize=11, fontweight='bold')
ax.set_title('Class Distribution — Imbalanced Dataset', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of objects', fontsize=11)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('\n⚠️  Observation: QSOs make up only 10% of the dataset.')
print('    A model that predicts STAR or GALAXY for everything could still achieve 90% accuracy.')
print('    This is exactly why accuracy alone is misleading on imbalanced data.')

In [ ]:
# --- Check 3: Outliers in temperature ---
print('=== TEMPERATURE OUTLIERS ===')
temp = df['temperature_K'].dropna()
mean, std = temp.mean(), temp.std()
lower, upper = mean - 3*std, mean + 3*std
outliers = df[df['temperature_K'] > upper]

print(f'  Temperature mean:  {mean:,.0f} K')
print(f'  Temperature std:   {std:,.0f} K')
print(f'  3σ upper bound:    {upper:,.0f} K')
print(f'  Outliers detected: {len(outliers)} rows (values > {upper:,.0f} K)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.hist(df['temperature_K'].dropna(), bins=60, color='steelblue', edgecolor='white')
ax1.axvline(upper, color='red', linestyle='--', linewidth=2, label=f'3σ = {upper:,.0f} K')
ax1.set_title('Temperature Distribution — Before Cleaning', fontsize=12)
ax1.set_xlabel('Temperature (K)'); ax1.legend()

clean_temp = df[df['temperature_K'] <= upper]['temperature_K'].dropna()
ax2.hist(clean_temp, bins=60, color='seagreen', edgecolor='white')
ax2.set_title('Temperature Distribution — After Removing Outliers', fontsize=12)
ax2.set_xlabel('Temperature (K)')
plt.tight_layout(); plt.show()

---
## Step 2 — Clean and Preprocess the Data

### 2.1 — The Data Cleaning Pipeline

From Step 1 we identified three problems:
1. Missing values in `u`, `g`, and `snr`
2. Extreme outliers in `temperature_K` (instrument errors)
3. Class imbalance — QSOs are rare (10%)

We will address each in turn. The reading described this as a structured process — automated validation, then standardisation. That is exactly what we are doing here.

> **Why not just drop missing rows?** Dropping rows loses data. With 4% missing, that could mean losing 200 rows. Instead, we impute — fill in a reasonable estimated value. For numerical features, the median is a robust choice (it is not affected by outliers the way the mean is).

In [ ]:
df_clean = df.copy()
feature_cols = ['u','g','r','i','z','redshift','temperature_K','snr']

# Step 1: Remove temperature outliers (instrument errors — replace with median)
temp_median = df_clean['temperature_K'].median()
outlier_mask = df_clean['temperature_K'] > (df_clean['temperature_K'].mean() + 3 * df_clean['temperature_K'].std())
df_clean.loc[outlier_mask, 'temperature_K'] = np.nan  # mark as missing first
print(f'Step 1 — Outliers removed: {outlier_mask.sum()} rows')

# Step 2: Impute all missing values with column median
for col in feature_cols:
    n_missing = df_clean[col].isnull().sum()
    if n_missing > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f'Step 2 — Imputed {n_missing:3d} missing values in "{col}" with median = {median_val:.4f}')

# Verify no missing values remain
assert df_clean[feature_cols].isnull().sum().sum() == 0, "Still missing values!"
print(f'\n✅ No missing values remain.')
print(f'   Dataset shape: {df_clean.shape}')

In [ ]:
# Step 3: Standardise all numerical features
# The reading explained why: different features have very different scales.
# Redshift ranges from ~0 to ~3; temperature ranges from ~3000 to ~18000.
# Without standardisation, the large-scale features dominate the model.

scaler = StandardScaler()
X = df_clean[feature_cols].values
y = df_clean['object_class'].values

print('Feature scales BEFORE standardisation:')
for col, mn, sd in zip(feature_cols,
                       df_clean[feature_cols].mean(),
                       df_clean[feature_cols].std()):
    print(f'  {col:15s}  mean: {mn:10.2f}   std: {sd:10.2f}')

print('\nFeature scales AFTER standardisation (mean ≈ 0, std ≈ 1 for all):')
X_scaled = scaler.fit_transform(X)
for col, mn, sd in zip(feature_cols, X_scaled.mean(axis=0), X_scaled.std(axis=0)):
    print(f'  {col:15s}  mean: {mn:+6.3f}   std: {sd:.3f}')

---
## Step 3 — Split the Data

### Why Stratified Splitting?

The reading explained that for imbalanced datasets, you must use **stratified splitting** — ensuring each split preserves the same class proportions as the full dataset. Without stratification, a random split might put all QSOs into the training set and none into the test set, making evaluation completely unreliable.

We use a **60% / 20% / 20%** split (train / validation / test), which is appropriate for a dataset of this size.

In [ ]:
# First split: 80% train+val, 20% test (stratified)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)

# Second split: 75% train, 25% val (of the 80%) → 60% / 20% overall
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

print('Split sizes:')
print(f'  Training:   {len(X_train):4d} samples  ({len(X_train)/len(X_scaled)*100:.0f}%)')
print(f'  Validation: {len(X_val):4d} samples  ({len(X_val)/len(X_scaled)*100:.0f}%)')
print(f'  Test:       {len(X_test):4d} samples  ({len(X_test)/len(X_scaled)*100:.0f}%)')

print('\nClass proportions in each split (stratification check):')
for split_name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    unique, counts = np.unique(y_split, return_counts=True)
    pcts = counts / len(y_split) * 100
    row = '  ' + split_name.ljust(7)
    for cls, pct in zip(unique, pcts):
        row += f'  {cls}: {pct:.1f}%'
    print(row)

print('\n✅ Class proportions are consistent across all splits — stratification is working.')
print('   This ensures evaluation is reliable even with the class imbalance.')

---
## Step 4 — Classification Metrics in Practice

### 4.1 — Train a Classifier

We will use a **Random Forest classifier** — a reliable, interpretable baseline that is well-suited to tabular data. The goal here is not to achieve the best possible accuracy; it is to use the trained model as a tool for understanding the evaluation metrics from the reading.

After training, we will compute the full set of metrics: accuracy, precision, recall, F1, and AUC.

In [ ]:
# Train a Random Forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

# Predict on the test set
y_pred       = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)   # probability for each class

class_names = clf.classes_
print(f'Classes: {class_names}')
print(f'Test set predictions shape: {y_pred.shape}')
print(f'Probability output shape:   {y_pred_proba.shape}  (one column per class)')
print('\n✅ Model trained and predictions generated.')

### 4.2 — Confusion Matrix

The confusion matrix shows exactly where the model makes mistakes. Each row is the true class; each column is the predicted class. Numbers on the diagonal are correct predictions; off-diagonal are errors.

From the reading: *True Positives (TP)* are on the diagonal; *False Negatives (FN)* are off-diagonal in the row; *False Positives (FP)* are off-diagonal in the column.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=class_names)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, linecolor='white', annot_kws={'size': 13})
ax1.set_xlabel('Predicted Class', fontsize=12)
ax1.set_ylabel('True Class', fontsize=12)
ax1.set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')

# Normalised (recall per row)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens', ax=ax2,
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, linecolor='white', annot_kws={'size': 13})
ax2.set_xlabel('Predicted Class', fontsize=12)
ax2.set_ylabel('True Class', fontsize=12)
ax2.set_title('Confusion Matrix (Normalised — Recall per Row)', fontsize=13, fontweight='bold')

plt.suptitle('Confusion Matrices — Stellar Object Classification', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Reading the normalised matrix (right):')
print('  Each row shows what fraction of true objects of that class were predicted as each class.')
print('  The diagonal = recall for that class. Off-diagonal = confusion with other classes.')

### 4.3 — Precision, Recall, F1, and AUC

Now we compute the full metric suite from the reading. For a three-class problem, precision and recall are computed **per class**, then averaged.

Recall the definitions:
- **Precision** = Of all objects I predicted as class X, what fraction actually are class X?
- **Recall** = Of all objects that truly are class X, what fraction did I find?
- **F1** = Harmonic mean of precision and recall — high only when both are high
- **AUC** = Probability that the model ranks a true positive above a true negative

In [ ]:
# Overall accuracy
acc = accuracy_score(y_test, y_pred)

# Per-class and averaged metrics
precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro    = recall_score(y_test, y_pred, average='macro')
f1_macro        = f1_score(y_test, y_pred, average='macro')
f1_weighted     = f1_score(y_test, y_pred, average='weighted')

# AUC (one-vs-rest for multi-class)
auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')

print('=== OVERALL METRICS ===')
print(f'  Accuracy:           {acc:.4f}  ({acc*100:.1f}%)')
print(f'  Macro Precision:    {precision_macro:.4f}')
print(f'  Macro Recall:       {recall_macro:.4f}')
print(f'  Macro F1:           {f1_macro:.4f}')
print(f'  Weighted F1:        {f1_weighted:.4f}')
print(f'  AUC (OvR, macro):   {auc:.4f}')

print('\n=== PER-CLASS REPORT ===')
print(classification_report(y_test, y_pred, target_names=class_names))

print('📊 Notice: is the QSO recall lower than STAR and GALAXY?')
print('   QSOs are rare (10% of the data) — the model has fewer examples to learn from.')
print('   This is why weighted vs macro averaging matters: they tell different stories.')

### 4.4 — ROC Curves

The ROC curve shows how the True Positive Rate (recall) varies against the False Positive Rate as the classification threshold changes — across all possible thresholds, not just the default 0.5. We plot one curve per class using a One-vs-Rest approach.

In [ ]:
from sklearn.preprocessing import label_binarize

# Binarize for one-vs-rest ROC
y_test_bin = label_binarize(y_test, classes=class_names)

fig, ax = plt.subplots(figsize=(8, 6))
colors = {'GALAXY': '#4CAF50', 'QSO': '#FF5722', 'STAR': '#2196F3'}

for i, cls in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
    cls_auc = roc_auc_score(y_test_bin[:, i], y_pred_proba[:, i])
    ax.plot(fpr, tpr, label=f'{cls}  (AUC = {cls_auc:.3f})',
            color=colors[cls], linewidth=2.5)

ax.plot([0,1],[0,1], 'k--', linewidth=1.2, alpha=0.5, label='Random classifier (AUC = 0.5)')
ax.fill_between([0,1],[0,1],[0,0], alpha=0.05, color='grey')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curves — One-vs-Rest\nStellar Object Classification', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('📊 The closer the curve is to the top-left corner, the better.')
print('   AUC = 1.0 would be perfect. AUC = 0.5 is random guessing.')
print('   Notice: which class has the highest AUC, and which has the lowest? Does this match the confusion matrix?')

---
## Step 5 — Why Accuracy Is Misleading on Imbalanced Data

This step directly demonstrates the concept from the reading: *"A model that predicts the majority class for everything can achieve very high accuracy while being completely useless."*

We create a dummy classifier that predicts the most common class (STAR) for every single input, regardless of the actual features. Then we compare its accuracy to its precision, recall, and F1.

In [ ]:
from sklearn.dummy import DummyClassifier

# Majority-class baseline: predicts STAR for everything
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_dummy = dummy.predict(X_test)

dummy_acc = accuracy_score(y_test, y_dummy)
dummy_f1  = f1_score(y_test, y_dummy, average='macro')
dummy_rec = recall_score(y_test, y_dummy, average='macro')

real_acc = accuracy_score(y_test, y_pred)
real_f1  = f1_score(y_test, y_pred, average='macro')
real_rec = recall_score(y_test, y_pred, average='macro')

print('=== ACCURACY TRAP DEMONSTRATION ===')
print(f'{"":30s}  {"Dummy (majority class)":>22}  {"Real model":>12}')
print(f'{"Accuracy":30s}  {dummy_acc:>22.4f}  {real_acc:>12.4f}')
print(f'{"Macro Recall":30s}  {dummy_rec:>22.4f}  {real_rec:>12.4f}')
print(f'{"Macro F1":30s}  {dummy_f1:>22.4f}  {real_f1:>12.4f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics   = ['Accuracy', 'Macro Recall', 'Macro F1']
dummy_vals = [dummy_acc, dummy_rec, dummy_f1]
real_vals  = [real_acc,  real_rec,  real_f1]

for ax, metric, dv, rv in zip(axes, metrics, dummy_vals, real_vals):
    bars = ax.bar(['Dummy\n(majority class)', 'Real model'], [dv, rv],
                  color=['tomato', 'steelblue'], edgecolor='white', linewidth=1.5, width=0.5)
    for bar, val in zip(bars, [dv, rv]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.grid(alpha=0.3, axis='y')

plt.suptitle('Why Accuracy Is Not Enough: Dummy vs Real Model', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'\n📊 The dummy model achieves {dummy_acc*100:.1f}% accuracy by predicting STAR for everything.')
print(f'   Its macro recall is {dummy_rec:.3f} — it finds 0% of QSOs and 0% of Galaxies.')
print(f'   Accuracy hides this completely. Macro F1 = {dummy_f1:.3f} exposes it immediately.')

---
## Step 6 — Interactive: Precision–Recall Trade-off

The reading explained that precision and recall conflict: raising one often lowers the other. This trade-off is controlled by the **decision threshold** — the probability above which the model declares a positive prediction.

The default threshold is 0.5. But you can change it:
- **Higher threshold** → model only predicts positive when very confident → fewer false positives → higher precision, lower recall
- **Lower threshold** → model flags more objects as positive → catches more true positives → higher recall, lower precision

Use the slider below to explore this trade-off for the QSO class (the rarest and most scientifically interesting class to detect correctly).

In [ ]:
# Get QSO probability scores from the model
qso_idx        = list(class_names).index('QSO')
qso_proba      = y_pred_proba[:, qso_idx]
y_test_qso_bin = (y_test == 'QSO').astype(int)

# Pre-compute full precision-recall curve
pr_precision, pr_recall, pr_thresholds = precision_recall_curve(y_test_qso_bin, qso_proba)

def explore_threshold(threshold=0.5):
    # Apply threshold
    y_qso_pred = (qso_proba >= threshold).astype(int)

    tp = ((y_qso_pred == 1) & (y_test_qso_bin == 1)).sum()
    fp = ((y_qso_pred == 1) & (y_test_qso_bin == 0)).sum()
    fn = ((y_qso_pred == 0) & (y_test_qso_bin == 1)).sum()
    tn = ((y_qso_pred == 0) & (y_test_qso_bin == 0)).sum()

    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Precision-Recall curve with current threshold marked
    ax1.plot(pr_recall, pr_precision, color='steelblue', linewidth=2.5, label='P–R curve')
    ax1.scatter([rec], [prec], color='tomato', s=180, zorder=5,
                label=f'Threshold = {threshold:.2f}\nPrecision = {prec:.3f}\nRecall = {rec:.3f}')
    ax1.set_xlabel('Recall', fontsize=12)
    ax1.set_ylabel('Precision', fontsize=12)
    ax1.set_title('Precision–Recall Curve (QSO class)', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.set_xlim(0, 1.05); ax1.set_ylim(0, 1.05)
    ax1.grid(alpha=0.3)

    # Metric bar chart
    metric_names  = ['Precision', 'Recall', 'F1']
    metric_values = [prec, rec, f1]
    bar_colors    = ['#2196F3', '#4CAF50', '#FF9800']
    bars = ax2.bar(metric_names, metric_values, color=bar_colors, edgecolor='white', width=0.4)
    for bar, val in zip(bars, metric_values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', fontsize=13, fontweight='bold')
    ax2.set_ylim(0, 1.2)
    ax2.set_title(f'Metrics at Threshold = {threshold:.2f}\nTP={tp}  FP={fp}  FN={fn}  TN={tn}',
                  fontsize=12, fontweight='bold')
    ax2.grid(alpha=0.3, axis='y')

    plt.suptitle('Interactive Precision–Recall Trade-off (QSO Detection)',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    print(f'At threshold {threshold:.2f}:  Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}')
    print(f'  The model flags {tp+fp} objects as QSO.')
    print(f'  Of these: {tp} are true QSOs ({prec*100:.1f}% precision).')
    print(f'  Of all true QSOs: {tp} found, {fn} missed ({rec*100:.1f}% recall).')

interact(explore_threshold,
         threshold=widgets.FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05,
                                       description='Threshold:', style={'description_width': '80px'}));

---
## Step 7 — Regression Metrics: Predicting Stellar Temperature

Classification metrics apply when the output is a category. When the output is a continuous number, we need regression metrics. The reading covered four: **MAE**, **MSE**, **RMSE**, and **R²**.

Here we train a regression model to predict a star's temperature in Kelvin from its photometric measurements — a genuinely useful astronomical task. We then compute all four metrics and interpret what each one is telling us.

> **Why does this matter?** Measuring stellar temperatures directly requires spectroscopy, which is expensive. If we can predict temperature reliably from photometry alone (which is much cheaper and more widely available), we can process far more objects.

In [ ]:
# Regression task: predict temperature from photometric features
reg_features = ['u','g','r','i','z','redshift','snr']
target_col   = 'temperature_K'

# Use only rows where temperature is within a physically sensible range
df_reg = df_clean[df_clean[target_col] < 30000].copy()

X_reg = df_reg[reg_features].values
y_reg = df_reg[target_col].values

# Scale
scaler_reg   = StandardScaler()
X_reg_scaled = scaler_reg.fit_transform(X_reg)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_reg_scaled, y_reg, test_size=0.2, random_state=42
)

# Train a Random Forest regressor
reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
reg.fit(X_tr, y_tr)
y_reg_pred = reg.predict(X_te)

# Compute all four regression metrics
mae  = mean_absolute_error(y_te, y_reg_pred)
mse  = mean_squared_error(y_te,  y_reg_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_te, y_reg_pred)

print('=== REGRESSION METRICS: Predicting Stellar Temperature ===')
print(f'  MAE  (Mean Absolute Error):        {mae:>10,.1f} K')
print(f'  MSE  (Mean Squared Error):         {mse:>10,.1f} K²')
print(f'  RMSE (Root Mean Squared Error):    {rmse:>10,.1f} K')
print(f'  R²   (Coefficient of Determination): {r2:>8.4f}')
print()
print(f'  Interpretation:')
print(f'    MAE:  On average, temperature predictions are off by {mae:,.0f} K')
print(f'    RMSE: Typical error (penalising large mistakes more) is {rmse:,.0f} K')
print(f'    RMSE > MAE ({rmse:.0f} > {mae:.0f}) → a few large errors are pulling RMSE up')
print(f'    R²:   The model explains {r2*100:.1f}% of the variance in stellar temperature')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted vs actual
axes[0].scatter(y_te, y_reg_pred, alpha=0.25, s=15, color='steelblue')
min_val, max_val = min(y_te.min(), y_reg_pred.min()), max(y_te.max(), y_reg_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2,
             label='Perfect prediction (y = x)')
axes[0].set_xlabel('True Temperature (K)', fontsize=12)
axes[0].set_ylabel('Predicted Temperature (K)', fontsize=12)
axes[0].set_title(f'Predicted vs True Temperature\nR² = {r2:.4f}', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Residuals (errors)
residuals = y_reg_pred - y_te
axes[1].hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0,    color='red',    linestyle='--', linewidth=2, label='Zero error')
axes[1].axvline(mae,  color='orange', linestyle=':',  linewidth=2, label=f'MAE  = +{mae:.0f} K')
axes[1].axvline(-mae, color='orange', linestyle=':',  linewidth=2, label=f'MAE  = -{mae:.0f} K')
axes[1].axvline(rmse, color='green',  linestyle='-.',  linewidth=2, label=f'RMSE = +{rmse:.0f} K')
axes[1].axvline(-rmse,color='green',  linestyle='-.',  linewidth=2, label=f'RMSE = -{rmse:.0f} K')
axes[1].set_xlabel('Residual (Predicted − True) in K', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Residual Distribution\n(Ideal: centred at 0, symmetric)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle('Regression Evaluation: Predicting Stellar Temperature from Photometry',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('📊 Notice: RMSE is larger than MAE. Look at the residual histogram.')
print('   Are there any predictions with very large errors (long tails)?')
print('   Those outlier predictions are what push RMSE above MAE.')

---
## Step 8 — Guided Reflection and Exercises

Answer the questions below by replacing the placeholder text. These are thinking prompts — use your observations from the cells above to answer them.

---

### ❓ Question 1 — Data Quality
**In Step 1, you found three data quality problems. Why did we replace outliers with NaN before imputing with the median, rather than replacing them directly with the median? What would go wrong if we used the mean instead of the median for imputation?**

> **Your answer:** *(Replace this text)*

---

### ❓ Question 2 — Class Imbalance and Accuracy
**In Step 5, the dummy majority-class model achieved high accuracy. What accuracy would you expect a dummy model to achieve if the dataset were 99% STAR and 1% QSO? And what would its recall for QSOs be? Why does macro F1 expose the problem that accuracy hides?**

> **Your answer:** *(Replace this text)*

---

### ❓ Question 3 — The Precision–Recall Trade-off
**Use the interactive widget in Step 6. At what threshold does QSO recall drop below 0.5? At what threshold does QSO precision rise above 0.9? If you were building a system to compile a clean catalogue of QSO candidates for expensive follow-up spectroscopy, would you prioritise precision or recall — and why?**

> **Your answer:** *(Replace this text)*

---

### ❓ Question 4 — Regression Metrics
**In Step 7, RMSE was larger than MAE. What does this tell you about the error distribution? Look at the residual histogram — are there many predictions with very large errors, or are most errors small and similar in size? When would you prefer MAE over RMSE as your primary metric?**

> **Your answer:** *(Replace this text)*

---

### ❓ Question 5 — Connecting to the Reading
**The reading says: "Most ML projects fail due to data issues, not algorithm choice." Based on what you did in Steps 1–3, name two specific data problems that — if left uncorrected — would have degraded the model's performance. How would each have affected the metrics you computed in Step 4?**

> **Your answer:** *(Replace this text)*

---
## ✅ Notebook Complete

You have:

| ✅ | Achievement |
|----|-------------|
| ✅ | Diagnosed three real-world data quality problems: missing values, outliers, and class imbalance |
| ✅ | Applied a full preprocessing pipeline: imputation, outlier handling, standardisation |
| ✅ | Split data with stratification to preserve class proportions across all splits |
| ✅ | Computed precision, recall, F1, AUC, and interpreted them using the confusion matrix |
| ✅ | Demonstrated concretely why accuracy alone is misleading on imbalanced data |
| ✅ | Explored the precision–recall trade-off interactively by adjusting the classification threshold |
| ✅ | Computed and interpreted MAE, RMSE, and R² on a continuous stellar temperature prediction task |

---

### ➡️ Next Step

Read the **Overfitting, Regularisation, and Hyperparameter Tuning** lesson pages before opening the Notebook 2 activity. Those pages explain the concepts you will explore hands-on in Notebook 2 — overfitting, dropout, learning rate, and regularisation strategies.